# CropScan: lab -> field fine-tuning (Step 4)

**Before running:** Settings -> Accelerator **GPU (T4 or P100)**, Internet **On**.
Then **Run All**. About 1.5-2.5 hours for 6 runs. When it finishes, download
`cropscan_runs.zip` from the Output panel and give it to Claude.

Same recipe as the original CropScan notebook (EfficientNet-B0, 5 head + 15 full
epochs); only the training data and augmentation change. Seed 42.

In [ ]:
import torch, subprocess, sys
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "scikit-learn"], check=True)

In [ ]:
# Embedded copies of scripts/train_mixed.py and experiments/manifest.json
import base64, gzip, pathlib
pathlib.Path("/kaggle/working/train_mixed.py").write_bytes(gzip.decompress(base64.b64decode("H4sIAGnMwmoC/7VaW3PbuJJ+169A8dTWkDMyLfmSzSpHU+s4nkx2Mzmp2JU86KhYMAlJHPN2QMq24ui/79cN8CbJ2ZmpXT/IJNBodDf6DjqOc12pQpwJ97U3EYs4U0fVOlPiarGIw1hl1QdVHb0eiTwTibwdAkIlkci1SONHFYlIVtIfDAT+ik21AlQZ6rioyuNKyzgLGMovNuLoiEAJxzdG8c2sPzqS66W4laUyw+KfjIv+jo5SmcULVVZCPRZKxymoKY/rQf/3ErsZtEc6zyvx9yjW4iGuVqJIZFY9xEl0bB6jPDz+GbD52kD9PBh8UmFcKBGX4lLnxXUosx/KoVhn4UpmSxVNxLtULhUxX2jFvIDaHaEMxRusBVJ35J95g5/Ee0BJLVZKRq9EsQJXYsxvEF+yGYpzoYo8XGGji0imX8RYHZ0ORZiXWFcvOBEySQaJ3CgNuPGBJWfi6Gf6/6Je6ot/AD2fhZBZJCBSkpWsYkjoXuoNn1B7AiTdjySXz3GSgEvB/Ak3lEWhoqEolRK1mL12pTkgs/ILhGuX/WRG3uRhjSdlPCJMZFmqknnvoDEHL27zasWDtQIQVdVKQbfiZZzJRGR5pW7z/O6HssdRs8jSIz6B5zz9pMr4q4roNMFBpXMco4aAknytxe9xVSk9FLfJGr86N5jwxEuF0hJyXA4Gl0SyiNNbCZZCNRFfVLxcVSoye1zLtEiUVbLxMTN4FObrrBIPDFj6g9/ySCUQYaJC2mMiblmD6RDF7QbHgckw10pMRapk5vYOgmZlGNZmZl+94YAEU8pUCb1OlFjA/tS90hu8Zq9ERTuUqoKcgLfK1+FKkcqFaihkxUJVWeQPrggf1LnQeahK4ngijNjck5OzocCPh+OsNR8HoNOhiCNIPg6xtMoZV6SKJN9gh4uP7/yB4ziDwUJDikGwWFdrrYIAEixyXUEZMyvqcjCox/SykLpU9XuYF5v6mUy6fjZHU79VsH6zC07Uirast7mkE1C6ns/Ctdag2TfkNGA3Kw1T/JjnydWjCtdVblcUslol8W0N9hGvDbXZOoXzkqXIioaUXIer3oufZQySGXwf372vcbEozaiBXFdxUvpsBxbkDZ7f5zIi9aRnnOPwsNp18NzHJdm2RZGSysFDwPyyErqRQtq/XV18GIrrmzfQsxnc08vzocC/8xf8b/RiPqThk5P/oHc+ffp3Ph9cXlz+ehVcv3tzhZUnL18K8TfsAzaMdkUqxG5Gu9jZ3KmiMloGX1quiJ4SKjO4/vUfn26A4slhF3xvNNyZCOfjZ2conMZH89CXZgi+mkfeONvBYBCpBbmjKIBTdOlhIuKs8sgDfsgzNeFoYVTFp2mG8V7huPwDo+YM4NrWMgn2J8J1JLuzzaaepcS6IfcuzkAInIxnCIgXgobEdCoccmVgAFJLExME/bjYZAidMNhbGEPaRDgElrXOOsfmX+Y40VK5swaGvWILYM3VPaGDxI/nDXvTzDP5QLJo+I3n0DDcr3C1X3O41eSXJC7cQ6g+K822/yzAJ2vg7vj8+e0u4YX1f7EPdm81qXYG/zOFxlEIywBZVuatlBAJIzTvq7XCw/h51Df5jcrKXO8Q9wG/MiFRNYbgzb3BH5X6PpttdDG+soRQ1BSB/xSGNfZHXofE/19Bn44Ob/WskM96Qj7bEfJZLeTR+fdYuCiKZOPOOuNv5RoxRGavEVNdCKGMl6kkiYyhmSQQuJgCeE8Po/0TB/cdsq5M5HZpo5PzzrGMTsijQSnp2AeDq88X74ObX+CPDp37IQOrA2KfvD+nbtZ/rXSc3blxurRZpc+/7MQ679aVpEsQiV8fZwaPUbnOp7evHaO6JaY6/vkYuVRGeH2iuafdNKgtK6l8dMeU9ayziKEf4gj5y4+iJOb2Zlcce8w05g2FEAsFIYjaf/3u/bsPVxefPHJ7pfg7UlxEH0VbgmHOicyi0rXxzHpJkkUQxFlcBQHcarJAblGplALXwoIwk5jx7QQ/VwuSSA3J8YheJmLmmlCLvYaU16rEmw86WyUqszt10FsBYc5td/K6y5aqosGWyM5qMDkUG9DTrp3F813klmwSJyS4sXqQIMoHJYRYuXV2PRFRHFYk/RzPlHkMOY2H0Di+QAIEb5+TOKVnBEDxjcMfDOw+CCXSPYPIEqrzB9KUepOZxTifMbL5bMKI5nR+/GTO7znwOsLZYQ5y/ZDesM8JioVr0i5m2w42kKuFkQTIdCh9dViZzAiGkKg6hiqHQ6jTHsCiXYvkEsJoZdCLEfXorAafY68uLc0hzPrszGcOllggZ26PYFqj8ZpdIhLy/jbN/JKEBb1MbyMpNE4IqoLkQc9Gcw+bsN46c9LnLP+XnIirfz8d82JivGWGa6MpqwhZ/N4pOTTjPL+tMeC8gMIzqmPBFBzYl8ua/SzZfelReqsee2ZAzCdxWbnqETlT4bYbWn8HOlxNHoTU0et5p5nLdmSkMZ57Hlc0PKbpRL8iBtIWdi3lqkQzQzEADc+tWd2ukUQGnAO7WZseZpmPUgzVkqE6JYPgPNlXdRWfqSq4Hbm2dJva6W6RH7weBV9sZffut4u3Vx+ubsb/HXweG3ZSn91djGpNAz+2vFb/WlO5JEFL5tftgcLE2TiDooVqeqPXCnIBgGkYuF08kIcfZ8FCSS5eANaXXQq2/9OWHXmw1DJyPZYDCjvyAS6z0bjWSN3HVAci8wxX0/HJS+sjGMpXMDTXoKfeCEoFo0d8HCxo6oa4oyH7S+MqLa6OU3yksMoUlZUM79yZjbbs/9qjDQihdZmTGIUm45nPPb/KXUNna1ysiwaphBKGSF4sjF9tCjCkMnmbqGjaGWTXQUm84/V9AXjzqSeBCGfpzBcVBT6jNY+ev4BnqCiojz0/LNau53Pl5/Zlj6qCyktZYfNKuUDbxHgq6Ut3MyGYLJJayw28c/eVtZJOyNDGvrK8S3D+mZ+qSsdh4zIX44ARMiCKkPwBFRdiDua4ZqncDWTG1udZCun0ER7z4pScko8qmwM7qRm/0Vr3qCAGZ5OhmJyag9a0jlSXgECka3fzZgWB2bd5i2/elceTkzlscS6px4aM3QEJYwyafMKI1TWbTMXG87nn4SG7GI+Qx4qTncyelp/uLjdcYTmRRHEPKiOzDaj5X9GlMtR5sNilqBawS4fEkuP0oZxahsE5ci/4zanBgOr0qwKiKDaV95TyfbFbYDlQATpHHFJwSAz6e3Koq129zgLuFtWGnNjmQF4gUQh1XLVGzdmu7lk0h0tr0hVKhgSHC061Cjkq5GUZlGtyhqPG0B8po4Fpmo06Zm0yncfWPKFMeRbcJnmIcndZ+7HN9wEafGDAZymy04IWQ0gB4NUO3P+p6Sf5Mq7K2vnD1HdmS5ojqboGEuy0IEa+Pv9zCdbzb+HgHqSG033VTFeqcMFcO7IuIvIPLaJG7j+ZM+DMkY/fGM4re1Y0bQfqQ8OQJa01QqtDQAhHZVB1zbLZ7digHTbI7IBVtZRVpd9LkQXkUffo/Au9XFPH4yO9abuLhK+IokDaOdcxjV3YSLjKcSDldObAnKijw21MeuCmLyVTGgEyhhl0zvwAPrle9tBxU6XB94fR1MkSlpKSTE2C/cfWNvcKf2UxIsNfWUZXBUem318vh3Mle1/IdVJNz59duVgnyfdWjp9fylH44KIXZ88uesj1ndKH93p+FVcbB9eYamalkmLqFEimOJUesnsq0/xOcY+7dJ7FTF26w8Sc2DWaM1YsZc2mxaVry76m10eDvmn4mXKQ/EuT3JhX17gZKkQ6XcO4DOS9jBNyTDApU7qkRdmBI8+BDKT0MXwYHomH5bC5+ZpyT9wn11wa8pr7L0rTg0o9VnWSkgX1jYvxIm19Y8cd2wHL6CJhKhbOE2MkTd8emWcY3tbp5IM8SI/HvOoVp1LpXRRrF3KkCznWaPjkR2QjQX7XUfBCU2awcGZPtHQ7r+X5ZP5vcV6LZF2u7JJBt2oz9Sy1j7cD40Bl1FY1dEc1OVRPD0XDUUC2O2RQO8ra15bMhyqgMl/rkAX4xC5sInaLw8YJNVPcw+52r+f9rIDTEHZ/e9h6bfAeiu2sYcRkXNWI9DBOEe7wUwd4vm9D1o5imEwlojBe88B5d1UHdjca1rW0h1LKIL3n9OApmrQgVHl7da3F+Nw/SLNnTooNdRcrl/jeX8a5p0mcqtQXkE+k7PzoTYZbQv/UEZQ4guwm/mixLV8Jp3Myi9osaAUvnxhUTxsmNLBpEU9tvYPqynd/3JUzV1DugaXmqGyRSYc19kcwJ7N0tpnvrzCHs2wcz1uVIRmlStzrXV7suKvS3k5OD18f1XXukC61AgNcTlvZUZDi+pR8qrXqZb3zdOk1dshbtNdWrm32MZZhc0dSuxOvLhgD6kZOeZjfhzXBU/t/z2yaPyLYRhyDwL7AmOkbA5XmenM4G3weJ6JMCZcFUg+iFj+L0Q7/TYvwng6AywdOKjsZpwxDq/qdyqdTlN/XubvXT+aa8mvWV6D7uVeXCt4uJ2xMQMlwMjHNSNdaDAPwHTLSR1DV9VVzm3ZiUW+Y7n5bUHZDe5A86nnQ33/uSdZ9DulBFLs905F/jt3a/RuPCwRmjvkZEsigLXjAX68BVMfAvaYC5fimR3OpkR9fZZXOi817PFpnanL3xuigk/5bFCrXPOx+v+ZgBCuoExRxyDf+5jcoKxQC3FcZiiMYvqlfjVOjzz3YIbgOJX6ODVP0HNQffZgvREY+1YiuQ1leDUbPXbAz/n1hC3TzvUG3yuPtYPtYg5oS9aRCjEypI5NZUjp9XIKnCVNT1r0oSp3gf+HnILOdbqtvM9ySqzvqQfDXLCQiprpbBTYyxnOc+vxhizsrdrft7kZp1M4ec2JjSpwYxxZEKpSbKYmiW8GtuL/R3S/RAQ+vqVS75C9oLrJMSbrdeP/J5Tr7JoBxTu3dCMnMawQ2tf+9nriCtlvG4H3xmONAKTfu32PCJVSSmrt/vujvIWJ+TCHan7iHctwbr7TrtXpwVnfrLtmTw9QgY+H/FJJX5l7bqpFJJQKqNJsuB3Fz5jVz2LWdkvvdkt0/pyGwWUbUE8Yff3xa8PRTtG0mGWXtBaXxgmHYuMFtn8FU6ju6bHB6o9Cqe3L1ZKuTPep27XhYYyG66LsVlASqoAcjUp+hAnb43AsTP/a328tjVCGeWMKT0Um0FU8s3a1pSzxVycQ/XWzrLKeS5vUbp21P9+XEP8Orc0CoP2Frx/89R3mPJJ+/xJhFc+A3KJxnpUbemng8kOoYDk3O3bLZCqfuQJSoujh/hk7jrIhRSIIbBJM2O8cQ1fh2BI8Y4BrOjtBzo0nm8xNMdZKVHa4dJmRfg8wRslZaFcdU46hNXoq0b1vnmflt2RYdVkqubaXHUelxjkYZbi2xVmuocfK9HnyrjxvRi/htyN80HfJ5Nz6SSGeG1jnOkTrBtt2MBKgFZOrNPJCXFKJcbMOZyPjFLhwUI9oG4MmZ98gpK+3GXnsDAIj54bUbXrnhOdfUiI4hlr/NdDz/AY5LmRqVy9hojdrXNTB0HUJflk1P6vK18Et5r74GYZ5CjgjiUY2VWA94az8rvjrkEPitbnNySx9r4bdY8gFpHX1FVN0HO7c9VF60+guYfcvdd1OO6Q4T9H5NDYStqm+HwtJMmucXlfNMLXxzdX0DKyVL/daxVXICevYDdY9/mG//Tbi/jHmg7mNj0GvM11yF9dSjMeQuEz1bHsR0x01UBAGH5yCgPmAQ2Htc0xQc/A/RE7TQGSwAAA==")))
pathlib.Path("/kaggle/working/manifest.json").write_bytes(gzip.decompress(base64.b64decode("...")))
print("files written")

In [ ]:
# Datasets: PlantWild v1 (HF), PlantDoc (GitHub). PlantVillage loads from HF inside the script.
import os, zipfile, urllib.request, shutil
from huggingface_hub import hf_hub_download
DATA = "/tmp/data"; os.makedirs(DATA, exist_ok=True)
zip_pw = hf_hub_download("uqtwei2/PlantWild", "plantwild.zip", repo_type="dataset")
zipfile.ZipFile(zip_pw).extractall(f"{DATA}/plantwild")
urllib.request.urlretrieve("https://codeload.github.com/pratikkayal/PlantDoc-Dataset/zip/refs/heads/master", "/tmp/pd.zip")
zipfile.ZipFile("/tmp/pd.zip").extractall("/tmp/pd")
shutil.move("/tmp/pd/PlantDoc-Dataset-master", f"{DATA}/plantdoc")
# 6 PlantDoc files differ only by letter case (CAR1.jpg / car1.jpg). The manifest was
# built on macOS, which cannot hold both, so the second copy is named *__casedup.
import json
for s in ("train", "val", "test"):
    for p, _ in json.load(open("/kaggle/working/manifest.json"))["plantdoc"][s]:
        if "__casedup" in p:
            shutil.copy(f"{DATA}/plantdoc/" + p.replace("__casedup", ""), f"{DATA}/plantdoc/" + p)
from datasets import load_dataset
for s in ("train", "test"):
    print("PlantVillage", s, len(load_dataset("GVJahnavi/PlantVillage_dataset", split=s)))

In [ ]:
# Every file the manifest names must exist before any GPU time is spent.
import json, os
m = json.load(open("/kaggle/working/manifest.json"))
for ds in ("plantwild", "plantdoc"):
    root = os.path.join("/tmp/data", m[ds]["root"])
    missing = [p for s in ("train", "val", "test") for p, _ in m[ds][s] if not os.path.exists(os.path.join(root, p))]
    print(ds, {s: len(m[ds][s]) for s in ("train", "val", "test")}, "missing:", len(missing), missing[:3])
    assert not missing

In [ ]:
# The 3 x 2 grid. Each run prints one line per epoch and its test scores at the end.
import subprocess, sys
for data in ("lab", "field", "mixed"):
    for aug in ("base", "field"):
        subprocess.run([sys.executable, "/kaggle/working/train_mixed.py", "--data", data, "--aug", aug,
                        "--manifest", "/kaggle/working/manifest.json", "--data-root", "/tmp/data",
                        "--out", "/kaggle/working/runs", "--workers", "4"], check=True)

In [ ]:
import json, glob, shutil
for f in sorted(glob.glob("/kaggle/working/runs/*/result.json")):
    r = json.load(open(f))
    print(f"{r['run']:13}", " | ".join(f"{d}: {t['top1']}% F1 {t['macro_f1']}" for d, t in r["test"].items()))
shutil.make_archive("/kaggle/working/cropscan_runs", "zip", "/kaggle/working/runs")
print("download cropscan_runs.zip from the Output panel")